# cap10_training_objective_handoff_v1

Purpose: turn the latest cap10 objective screen failure into a concrete, non-promotional training-objective handoff. The previous notebook showed that a low-floor participation controller improves 2023-2024 underparticipation but still fails both the `+30pp` relative return target and the `<=15%` MaxDD budget. This notebook decides what the next training/checkpoint objective must optimize and what mechanisms should stop.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer family.
- Direct comparison baseline: `sparse_release_cap10_rebound_balance_v1` and `cap10_state_conditioned_trend_participation_controller_v1`.
- Role: `diagnose` / `training-objective handoff`.
- Changed layer: `model_selection` and shared `training_objective_handoff` interpretation.
- Target: preserve notebook-first evidence while moving reusable handoff logic into `src/soxl_quant/app/`.
- Acceptance: no runtime promotion; output a clear next objective with required terms, blocked mechanisms, leakage guard, and fold-local target requirements.

External basis:
- Qlib reproducible AI quant workflow: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PBO/CSCV and Deflated Sharpe as guards against selecting from repeated screens: https://doi.org/10.21314/JCF.2016.322 and https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551
- Purged / embargoed financial time-series validation for future target labels: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2708678
- Conservative/offline RL motivation for not over-trusting OOS-derived labels before fold-local validation: https://arxiv.org/abs/2006.04779
- CVaR/risk-sensitive RL motivation for explicit downside state loss rather than return-only tuning: https://arxiv.org/abs/1506.02188


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "AGENTS.md").exists() and (path / "src/soxl_quant").exists():
            return path
    raise RuntimeError("could not locate repository root")


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.training_objective_handoff import (
    derive_training_objective_handoff,
    render_training_objective_handoff,
)

EXPERIMENT = "cap10_training_objective_handoff_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
SOURCE_OBJECTIVE_CARD = ROOT / "runtime/model_selection/card/cap10_state_conditioned_trend_participation_controller_v1"
SOURCE_DIAGNOSTIC_CARD = ROOT / "runtime/model_selection/card/cap10_release_oos_poor_window_diagnostic_v1"
CARD.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


## 1. Load Latest Evidence

This cell treats already-executed runtime cards as authoritative evidence. It does not recompute daily paths or mutate market data.

In [2]:
objective_scores = pd.read_csv(SOURCE_OBJECTIVE_CARD / "objective_scores.csv")
objective_selection = json.loads((SOURCE_OBJECTIVE_CARD / "selection.json").read_text())
poor_window_clusters = pd.read_csv(SOURCE_DIAGNOSTIC_CARD / "aggregate/poor_window_clusters.csv")
feature_groups = pd.read_csv(SOURCE_DIAGNOSTIC_CARD / "aggregate/feature_group_consensus.csv")

context = {
    "source_objective_card": str(SOURCE_OBJECTIVE_CARD.relative_to(ROOT)),
    "source_diagnostic_card": str(SOURCE_DIAGNOSTIC_CARD.relative_to(ROOT)),
    "objective_candidates": int(len(objective_scores)),
    "poor_window_clusters": int(len(poor_window_clusters)),
    "feature_groups": int(len(feature_groups)),
    "source_decision": objective_selection["decision"],
    "source_selected_candidate": objective_selection["selected_candidate"],
}
print(json.dumps(context, indent=2))
display(objective_scores)
display(poor_window_clusters)
display(feature_groups.head(6))


{
  "source_objective_card": "runtime/model_selection/card/cap10_state_conditioned_trend_participation_controller_v1",
  "source_diagnostic_card": "runtime/model_selection/card/cap10_release_oos_poor_window_diagnostic_v1",
  "objective_candidates": 9,
  "poor_window_clusters": 3,
  "feature_groups": 5,
  "source_decision": "diagnostic_no_objective_pass_no_promotion",
  "source_selected_candidate": "low_floor_rebound_dd10"
}


,candidate,window_count,window_pass_count,window_gate,cluster_count,cluster_gate,objective_pass,objective_loss,relative_loss,drawdown_loss,participation_loss,min_relative_return,max_max_dd,mean_avg_pos,worst_window,worst_cluster
0,low_floor_rebound_dd10,12,0,False,3,False,False,1137.826042,937.583511,168.759616,31.482915,-143.579076,22.022415,0.135877,cap10_release_0.25__wf_2023_2024,C003
1,low_floor_rebound_dd08,12,1,False,3,False,False,1200.406474,1051.545357,107.900185,40.960932,-166.067266,21.157998,0.113105,cap10_release_0.25__wf_2023_2024,C003
2,soxx_te_low_floor_dd10,12,0,False,3,False,False,1261.265983,1079.742993,139.283292,42.239697,-170.624900,21.073647,0.116506,cap10_release_0.25__wf_2023_2024,C003
3,no_overlay_replay,12,4,False,3,False,False,1282.887653,1182.127484,48.931259,51.828910,-193.103999,18.678084,0.082236,cap10_release_0.15__wf_2023_2024,C003
4,soxx_strength_floor30_dd10,12,0,False,3,False,False,1352.209117,1030.383715,286.375276,35.450126,-162.749881,23.385024,0.125655,cap10_release_0.10__wf_2023_2024,C003
5,permissive_floor35_dd12,12,0,False,3,False,False,1357.120455,884.127742,440.169433,32.823280,-135.258465,25.405292,0.145821,cap10_release_0.10__wf_2023_2024,C003
6,trend_floor35_dd08,12,1,False,3,False,False,1401.108836,1064.370438,301.388531,35.349867,-170.649171,23.114103,0.122230,cap10_release_0.15__wf_2023_2024,C003
7,strict_soxx_te45_dd15,12,2,False,3,False,False,1463.039688,1196.961422,219.159324,46.918942,-195.803149,21.875399,0.095814,cap10_release_0.15__wf_2023_2024,C003
8,trend_floor45_dd08,12,1,False,3,False,False,1562.530040,1096.128430,418.985175,47.416435,-175.076852,22.812781,0.105903,cap10_release_0.15__wf_2023_2024,C003


,cluster_id,cluster_start,cluster_end,profile_count,window_count,profiles,mean_excess_return,worst_excess_return,mean_max_dd,max_max_dd,mean_avg_pos,mean_bad_score,max_bad_score,dominant_failure_type,primary_optimization_path
0,C003,2023-10-23,2024-03-07,6,48,"cap10_release_0.10__wf_2023_2024,cap10_release...",-112.967255,-130.537576,5.151819,5.151819,0.106834,142.967255,160.537576,underparticipation,reentry_participation_or_behavior_clone
1,C002,2022-09-02,2022-12-29,6,33,"cap10_release_0.10__wf_2022,cap10_release_0.15...",-8.070640,-17.893409,6.808061,9.060112,0.018385,38.070640,47.893409,underparticipation,reentry_participation_or_behavior_clone
2,C001,2022-05-09,2022-08-18,5,15,"cap10_release_0.10__wf_2022,cap10_release_0.15...",-4.818807,-10.913696,8.937712,14.739362,0.080653,34.818807,40.913696,underparticipation,reentry_participation_or_behavior_clone


,feature_group,feature_count,mean_abs_delta_z,max_stability_score,mean_spearman_to_excess,top_features
0,regime_or_drawdown,6,0.947317,4.694082,-0.017558,"rolling_new_low_ratio_63,regime_risk,drawdown_..."
1,volatility_or_downside,12,0.870869,4.478630,0.079657,"realized_kurtosis_21,vol_coef_21,regime_risk_n..."
2,trend_or_return,12,0.851141,3.490700,-0.057655,"anchor_slope_180,below_anchor_ratio_63,anchor_..."
3,soxx_relative_or_tracking_error,3,0.802526,2.467018,-0.008355,"tracking_error_vol_21,relative_strength_21,tra..."
4,other,7,0.695407,1.608011,-0.107401,"cum_drop_30,rsi_14,cum_drop_20,days_since_high..."


## 2. Handoff Derivation

The shared helper now understands both older walk-forward selector rows (`full_gate` / `weighted_objective_loss`) and objective-screen rows (`objective_pass` / `objective_loss`). That matters here because the cap10 screen is explicitly a joint participation/drawdown failure, not a simple threshold router failure.

In [3]:
handoff = derive_training_objective_handoff(
    walkforward_scores=objective_scores,
    poor_window_clusters=poor_window_clusters,
    feature_groups=feature_groups,
    router_selection=objective_selection,
)
review = render_training_objective_handoff(handoff)

(CARD / "training_objective_handoff.json").write_text(json.dumps(handoff, ensure_ascii=True, indent=2), encoding="utf-8")
(CARD / "training_objective_handoff.md").write_text(review, encoding="utf-8")

display(Markdown(review))
print(json.dumps(handoff, indent=2))


# Training Objective Handoff

## Decision

- decision: `train_state_conditioned_multi_objective`
- promotion: `blocked`
- selected_diagnostic_candidate: `low_floor_rebound_dd10`

## Objective Terms

- `walkforward_relative_return_gap_penalty`
- `walkforward_drawdown_gap_penalty`
- `high_vol_rebound_participation_floor`
- `downtrend_drawdown_state_cap`

## Risk Gap

- max_dd_gap=7.02pp, relative_gap=173.58pp, worst_window=`cap10_release_0.25__wf_2023_2024`

## Primary Underparticipation Cluster

- cluster=`C003`, start=2023-10-23, end=2024-03-07, mean_excess=-112.97pp

## Stop Mechanisms

- `path_level_threshold_router_search`


{
  "decision": "train_state_conditioned_multi_objective",
  "promotion": "blocked",
  "selected_diagnostic_candidate": "low_floor_rebound_dd10",
  "selected_dominant_failure_type": "joint_tradeoff",
  "risk_gap": {
    "target_max_dd": 15.0,
    "observed_max_dd": 22.02,
    "max_dd_gap": 7.02,
    "target_relative_return": 30.0,
    "observed_min_relative_return": -143.58,
    "relative_gap": 173.58,
    "worst_window": "cap10_release_0.25__wf_2023_2024"
  },
  "primary_underparticipation_cluster": {
    "cluster_id": "C003",
    "cluster_start": "2023-10-23",
    "cluster_end": "2024-03-07",
    "profile_count": 6,
    "window_count": 48,
    "profiles": "cap10_release_0.10__wf_2023_2024,cap10_release_0.15__wf_2023_2024,cap10_release_0.20__wf_2023_2024,cap10_release_0.25__wf_2023_2024,cap10_release_0.30__wf_2023_2024,cap10_release_0.35__wf_2023_2024",
    "mean_excess_return": -112.96725518906273,
    "worst_excess_return": -130.5375760374115,
    "mean_max_dd": 5.151819388506151,
 

## 3. Training Recipe And Stop Rules

The recipe below is intentionally a handoff, not a training run. The latest failure was selected from OOS / walk-forward daily paths, so it can define objective terms and checks, but direct labels must be regenerated fold-locally before BC/PPO/offline RL.

In [4]:
objective_terms = list(handoff.get("objective_terms", []))
risk_gap = handoff.get("risk_gap", {})
cluster = handoff.get("primary_underparticipation_cluster", {})
state_feature_groups = list(handoff.get("state_feature_groups", []))

recipe_rows = [
    {
        "component": "relative_return_gap",
        "required_term": "walkforward_relative_return_gap_penalty",
        "evidence": f"worst relative gap {risk_gap.get('relative_gap', '')}pp vs target +{TARGET_RELATIVE_RETURN}pp",
        "training_use": "checkpoint loss and PPO reward shaping; do not tune directly on held-out labels",
        "fold_local_requirement": True,
    },
    {
        "component": "drawdown_budget",
        "required_term": "walkforward_drawdown_gap_penalty",
        "evidence": f"max drawdown gap {risk_gap.get('max_dd_gap', '')}pp vs target {TARGET_MAX_DD}%",
        "training_use": "sequence-risk penalty and checkpoint loss",
        "fold_local_requirement": True,
    },
    {
        "component": "rebound_participation",
        "required_term": "high_vol_rebound_participation_floor",
        "evidence": f"primary cluster {cluster.get('cluster_id', '')} underparticipation, mean excess {cluster.get('mean_excess_return', '')}pp",
        "training_use": "sparse fold-local labels or supervised head; require post-event false-positive audit",
        "fold_local_requirement": True,
    },
    {
        "component": "downtrend_risk_cap",
        "required_term": "downtrend_drawdown_state_cap",
        "evidence": "post-hoc low-floor improved recovery but pushed 2022 MaxDD above budget",
        "training_use": "state-conditioned cap learned before entry, not path-end mechanical de-risk",
        "fold_local_requirement": True,
    },
]
recipe_df = pd.DataFrame(recipe_rows)
recipe_df["active_in_handoff"] = recipe_df["required_term"].isin(objective_terms)
recipe_df.to_csv(CARD / "training_objective_recipe.csv", index=False)

stop_rules = pd.DataFrame(
    [
        {
            "mechanism": "path_level_threshold_router_search",
            "status": "stop",
            "reason": "cap10 low-floor and SOXX/SOXL threshold overlays did not clear joint +30pp/<=15% gate",
        },
        {
            "mechanism": "direct_oos_label_training",
            "status": "blocked",
            "reason": "latest paths are OOS/walk-forward diagnostics; regenerate fold-local targets before BC/PPO",
        },
        {
            "mechanism": "direct_soxx_actor_feature_retrain",
            "status": "blocked",
            "reason": "SOXX/SOXL threshold gates did not pass; use only as candidate state features after IC/single-signal validation",
        },
    ]
)
stop_rules.to_csv(CARD / "stop_rules.csv", index=False)

display(recipe_df)
display(stop_rules)


,component,required_term,evidence,training_use,fold_local_requirement,active_in_handoff
0,relative_return_gap,walkforward_relative_return_gap_penalty,worst relative gap 173.58pp vs target +30.0pp,checkpoint loss and PPO reward shaping; do not...,True,True
1,drawdown_budget,walkforward_drawdown_gap_penalty,max drawdown gap 7.02pp vs target 15.0%,sequence-risk penalty and checkpoint loss,True,True
2,rebound_participation,high_vol_rebound_participation_floor,"primary cluster C003 underparticipation, mean ...",sparse fold-local labels or supervised head; r...,True,True
3,downtrend_risk_cap,downtrend_drawdown_state_cap,post-hoc low-floor improved recovery but pushe...,"state-conditioned cap learned before entry, no...",True,True


,mechanism,status,reason
0,path_level_threshold_router_search,stop,cap10 low-floor and SOXX/SOXL threshold overla...
1,direct_oos_label_training,blocked,latest paths are OOS/walk-forward diagnostics;...
2,direct_soxx_actor_feature_retrain,blocked,SOXX/SOXL threshold gates did not pass; use on...


In [5]:
fold_local_requirements = {
    "target_scope": "fold_local_required_before_training",
    "no_training_from_this_card": True,
    "purged_or_embargoed_validation_required": True,
    "minimum_checks_before_bc_or_ppo": [
        "audit_state_objective_target_scope eval_overlap_rows == 0",
        "audit_state_objective_target_lift current_positive_rebound_lift_rows > 0",
        "state-head or release-gate false-positive audit on non-overlapping event folds",
        "checkpoint-objective selection must include window_gate_loss and risk_cap_violation",
        "walk-forward replay after training must include wf_2020, wf_2022, wf_2023_2024 and fixed-split evidence",
    ],
    "candidate_feature_groups": state_feature_groups,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
}
(CARD / "fold_local_requirements.json").write_text(
    json.dumps(fold_local_requirements, ensure_ascii=True, indent=2),
    encoding="utf-8",
)
print(json.dumps(fold_local_requirements, indent=2))


{
  "target_scope": "fold_local_required_before_training",
  "no_training_from_this_card": true,
  "purged_or_embargoed_validation_required": true,
  "minimum_checks_before_bc_or_ppo": [
    "audit_state_objective_target_scope eval_overlap_rows == 0",
    "audit_state_objective_target_lift current_positive_rebound_lift_rows > 0",
    "state-head or release-gate false-positive audit on non-overlapping event folds",
    "checkpoint-objective selection must include window_gate_loss and risk_cap_violation",
    "walk-forward replay after training must include wf_2020, wf_2022, wf_2023_2024 and fixed-split evidence"
  ],
  "candidate_feature_groups": [
    "regime_or_drawdown",
    "volatility_or_downside"
  ],
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 15.0
}


## 4. Persist Selection

The notebook remains the primary single-run conclusion. JSON/CSV sidecars support future automation and retrieval.

In [6]:
def jsonable(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, list):
        return [jsonable(v) for v in value]
    if isinstance(value, tuple):
        return [jsonable(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_experiments": [
        "cap10_state_conditioned_trend_participation_controller_v1",
        "cap10_release_oos_poor_window_diagnostic_v1",
    ],
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "sparse_release_cap10_rebound_balance_v1",
    "role": "diagnose",
    "changed_layer": ["model_selection", "training_objective", "shared_handoff_logic"],
    "decision": handoff.get("decision"),
    "promotion_allowed": False,
    "training_allowed_from_this_card": False,
    "next_mechanism": handoff.get("decision"),
    "selected_diagnostic_candidate": handoff.get("selected_diagnostic_candidate"),
    "risk_gap": handoff.get("risk_gap"),
    "objective_terms": handoff.get("objective_terms"),
    "stop_mechanisms": handoff.get("stop_mechanisms"),
    "fold_local_requirements": fold_local_requirements,
    "external_sources": [
        "https://github.com/microsoft/qlib",
        "https://arxiv.org/abs/2009.11189",
        "https://doi.org/10.21314/JCF.2016.322",
        "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551",
        "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2708678",
        "https://arxiv.org/abs/2006.04779",
        "https://arxiv.org/abs/1506.02188",
    ],
    "sidecars": [
        "training_objective_handoff.json",
        "training_objective_handoff.md",
        "training_objective_recipe.csv",
        "stop_rules.csv",
        "fold_local_requirements.json",
        "summary.json",
        "selection.json",
    ],
}
summary = {
    "experiment": EXPERIMENT,
    "decision": selection["decision"],
    "promotion_allowed": False,
    "training_allowed_from_this_card": False,
    "selected_diagnostic_candidate": selection["selected_diagnostic_candidate"],
    "risk_gap": selection["risk_gap"],
    "objective_terms": selection["objective_terms"],
    "conclusion": [
        "The latest cap10 screen is a joint underparticipation plus drawdown-budget tradeoff, not a threshold-search near miss.",
        "Next valid work is fold-local state-conditioned multi-objective target generation / checkpoint selection, with direct OOS-label training blocked.",
        "Path-level SOXX/SOXL threshold overlays should stop; SOXX/SOXL remains only a feature hypothesis for IC/single-signal/fold-local validation.",
    ],
}

(CARD / "selection.json").write_text(json.dumps(jsonable(selection), ensure_ascii=True, indent=2), encoding="utf-8")
(CARD / "summary.json").write_text(json.dumps(jsonable(summary), ensure_ascii=True, indent=2), encoding="utf-8")

assert selection["decision"] == "train_state_conditioned_multi_objective"
assert selection["promotion_allowed"] is False
assert selection["training_allowed_from_this_card"] is False
assert "downtrend_drawdown_state_cap" in selection["objective_terms"]
assert "high_vol_rebound_participation_floor" in selection["objective_terms"]
print(json.dumps(jsonable(summary), indent=2))


{
  "experiment": "cap10_training_objective_handoff_v1",
  "decision": "train_state_conditioned_multi_objective",
  "promotion_allowed": false,
  "training_allowed_from_this_card": false,
  "selected_diagnostic_candidate": "low_floor_rebound_dd10",
  "risk_gap": {
    "target_max_dd": 15.0,
    "observed_max_dd": 22.02,
    "max_dd_gap": 7.02,
    "target_relative_return": 30.0,
    "observed_min_relative_return": -143.58,
    "relative_gap": 173.58,
    "worst_window": "cap10_release_0.25__wf_2023_2024"
  },
  "objective_terms": [
    "walkforward_relative_return_gap_penalty",
    "walkforward_drawdown_gap_penalty",
    "high_vol_rebound_participation_floor",
    "downtrend_drawdown_state_cap"
  ],
  "conclusion": [
    "The latest cap10 screen is a joint underparticipation plus drawdown-budget tradeoff, not a threshold-search near miss.",
    "Next valid work is fold-local state-conditioned multi-objective target generation / checkpoint selection, with direct OOS-label training block

## 5. Runtime Conclusion

This card closes the loop after `cap10_state_conditioned_trend_participation_controller_v1`: the next action is not another overlay sweep. It is a fold-local target-generation or checkpoint-objective loop that explicitly combines rebound participation with sequence-risk control.

In [7]:
selection = json.loads((CARD / "selection.json").read_text())
recipe_df = pd.read_csv(CARD / "training_objective_recipe.csv")
stop_rules = pd.read_csv(CARD / "stop_rules.csv")

conclusion = f"""
### Decision: `{selection['decision']}`

The selected diagnostic candidate remains `{selection['selected_diagnostic_candidate']}` from the cap10 objective screen, but this card blocks promotion and blocks direct training from the OOS-derived target rows. Risk gap: max-DD gap `{selection['risk_gap']['max_dd_gap']}pp`, relative-return gap `{selection['risk_gap']['relative_gap']}pp`, worst window `{selection['risk_gap']['worst_window']}`.

Required objective terms: `{', '.join(selection['objective_terms'])}`.

Stop rules: no more path-level threshold router search for this branch, no direct OOS-label BC/PPO, and no direct SOXX actor-feature retrain from this evidence. SOXX/SOXL spread remains a feature hypothesis only after IC/single-signal/fold-local validation.

Next concrete loop: generate fold-local state-conditioned multi-objective targets, run leakage/lift semantics audits, then score checkpoints or BC/PPO candidates by alignment + walk-forward gate loss before any promotion review.
"""
display(Markdown(conclusion))
display(recipe_df)
display(stop_rules)



### Decision: `train_state_conditioned_multi_objective`

The selected diagnostic candidate remains `low_floor_rebound_dd10` from the cap10 objective screen, but this card blocks promotion and blocks direct training from the OOS-derived target rows. Risk gap: max-DD gap `7.02pp`, relative-return gap `173.58pp`, worst window `cap10_release_0.25__wf_2023_2024`.

Required objective terms: `walkforward_relative_return_gap_penalty, walkforward_drawdown_gap_penalty, high_vol_rebound_participation_floor, downtrend_drawdown_state_cap`.

Stop rules: no more path-level threshold router search for this branch, no direct OOS-label BC/PPO, and no direct SOXX actor-feature retrain from this evidence. SOXX/SOXL spread remains a feature hypothesis only after IC/single-signal/fold-local validation.

Next concrete loop: generate fold-local state-conditioned multi-objective targets, run leakage/lift semantics audits, then score checkpoints or BC/PPO candidates by alignment + walk-forward gate loss before any promotion review.


,component,required_term,evidence,training_use,fold_local_requirement,active_in_handoff
0,relative_return_gap,walkforward_relative_return_gap_penalty,worst relative gap 173.58pp vs target +30.0pp,checkpoint loss and PPO reward shaping; do not...,True,True
1,drawdown_budget,walkforward_drawdown_gap_penalty,max drawdown gap 7.02pp vs target 15.0%,sequence-risk penalty and checkpoint loss,True,True
2,rebound_participation,high_vol_rebound_participation_floor,"primary cluster C003 underparticipation, mean ...",sparse fold-local labels or supervised head; r...,True,True
3,downtrend_risk_cap,downtrend_drawdown_state_cap,post-hoc low-floor improved recovery but pushe...,"state-conditioned cap learned before entry, no...",True,True


,mechanism,status,reason
0,path_level_threshold_router_search,stop,cap10 low-floor and SOXX/SOXL threshold overla...
1,direct_oos_label_training,blocked,latest paths are OOS/walk-forward diagnostics;...
2,direct_soxx_actor_feature_retrain,blocked,SOXX/SOXL threshold gates did not pass; use on...
